In [1]:
import cv2
import numpy as np
import mediapipe as mp
from scipy.signal import find_peaks, butter, filtfilt, detrend, savgol_filter, welch
from scipy.signal.windows import hamming
import pywt
import time
from collections import deque
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

# MediaPipe 초기화
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False,
                                  max_num_faces=1,
                                  refine_landmarks=True,
                                  min_detection_confidence=0.5,
                                  min_tracking_confidence=0.5)

def display_triangular_regions(frame, regions):
    """
    화면에 삼각형 영역을 표시하는 함수.
    """
    for triangle_points in regions:
        cv2.polylines(frame, [np.int32(triangle_points)], isClosed=True, color=(0, 255, 0), thickness=2)

def extract_green_signals_by_face_regions(frame, landmarks):
    h, w, _ = frame.shape

    # 필요한 랜드마크 인덱스 (MediaPipe Face Mesh 기준)
    # MediaPipe는 총 468개의 얼굴 랜드마크를 제공합니다.
    left_cheek_1 = [118, 119, 101]
    left_cheek_2 = [187, 50, 205]
    left_cheek_3 = [207, 214, 212]
    left_cheek_4 = [203, 165, 206]
    left_cheek_5 = [100, 36, 142]
    left_cheek_6 = [117, 111, 123]
    
    right_cheek_1 = [347, 348, 330]
    right_cheek_2 = [280, 425, 411]
    right_cheek_3 = [427, 432, 434]
    right_cheek_4 = [423, 426, 391]
    right_cheek_5 = [329, 371, 266]
    right_cheek_6 = [346, 340, 352]
    
    left_chin_indices = [194, 211, 32, 201]
    mid_chin_indices = [200, 83, 313]
    right_chin_indices = [418, 431, 262, 421]

    # ROI 영역 정의
    roi_landmarks = [
        left_cheek_1, left_cheek_2, left_cheek_3, left_cheek_4, left_cheek_5, left_cheek_6,
        right_cheek_1, right_cheek_2, right_cheek_3, right_cheek_4, right_cheek_5, right_cheek_6,
        left_chin_indices, mid_chin_indices, right_chin_indices
    ]

    green_signals = []
    regions = []

    for indices in roi_landmarks:
        # 삼각형 꼭짓점 좌표 생성
        triangle_points = np.array([[landmarks[idx][0], landmarks[idx][1]] for idx in indices])

        # 빈 마스크 생성
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [np.int32(triangle_points)], 255)  # 삼각형 영역을 하얗게 채우기

        # 원본 이미지에서 삼각형 영역 추출
        masked_roi = cv2.bitwise_and(frame, frame, mask=mask)

        # 피부 영역에 대한 Gaussian 블러 적용 (Spatial Filtering)
        blurred_roi = cv2.GaussianBlur(masked_roi, (7, 7), 0)

        # 삼각형 영역의 G 채널 평균 값 계산
        mean_g = cv2.mean(blurred_roi, mask=mask)[1]  # G 채널 값
        green_signals.append(mean_g)

        # 삼각형 영역의 좌표를 그대로 저장
        regions.append(triangle_points)

    return np.array(green_signals), regions

# 그린 채널 방식 적용
def green_channel_method(S):
    S = np.array(S)
    mean_g = np.mean(S)
    C = (S / mean_g) - 1
    return C

# 대역 통과 필터 적용 (필터 파라미터 조정)
def bandpass_filter(signal, lowcut=0.7, highcut=3.5, fs=30.0, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    print(f"Bandpass Filter - fs: {fs}, nyquist: {nyquist}, low: {low}, high: {high}")
    if low >= high:
        # 주파수 범위가 잘못되었을 경우 기본값으로 설정
        low = 0.7 / nyquist
        high = 3.5 / nyquist
    # low와 high가 (0, 1) 범위 내에 있도록 조정
    if low <= 0 or high >= 1:
        print("Adjusted low and high to be within (0, 1) range")
        low = max(min(low, 0.99), 0.01)
        high = max(min(high, 0.99), 0.01)
    b, a = butter(order, [low, high], btype='band')
    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal

# 신호 스무딩 함수 (사비츠키-골레이 필터 적용)
def smooth_signal(signal, window_length=5, polyorder=2):
    # 신호 길이가 윈도우 길이보다 짧으면 윈도우 길이를 조정
    if len(signal) < window_length:
        window_length = len(signal) if len(signal) % 2 == 1 else len(signal) - 1
    if window_length < 3:
        return signal  # 스무딩 적용 불가
    smoothed_signal = savgol_filter(signal, window_length=window_length, polyorder=polyorder)
    return smoothed_signal

# 심박수 계산 함수 - 피크 검출
def calculate_heart_rate_peaks(signal, fs):
    # 신호 디트렌딩
    signal = detrend(signal)
    # 신호 정규화
    signal = (signal - np.mean(signal)) / np.std(signal)
    # 신호 스무딩
    signal = smooth_signal(signal, window_length=5, polyorder=2)
    # 신호 미분
    diff_signal = np.diff(signal)
    diff_signal = np.insert(diff_signal, 0, 0)  # 길이 맞추기

    # 피크 검출 파라미터 설정 (min_distance 조정)
    min_distance = int(fs * 0.5)  # 최소 피크 간격 (0.5초)
    prominence = 0.4 * np.std(diff_signal)  # 피크의 중요도 설정

    # 피크 검출
    peaks, _ = find_peaks(diff_signal, distance=min_distance, prominence=prominence)

    # 피크 간격 계산
    peak_intervals = np.diff(peaks) / fs
    if len(peak_intervals) > 0:
        heart_rate = 60.0 / np.mean(peak_intervals)
        return heart_rate, peaks
    else:
        return 0, peaks

# 심박수 계산 함수 - FFT와 웨이블릿 변환 결합
def calculate_heart_rate_fft_wavelet(signal, fs):
    # 신호 디트렌딩 및 정규화
    signal = detrend(signal)
    signal = (signal - np.mean(signal)) / np.std(signal)

    # 해밍 창 함수 적용
    windowed_signal = signal * hamming(len(signal))

    # 제로 패딩하여 신호 길이 확장 (다음 2의 거듭제곱)
    n = len(windowed_signal)
    n_padded = 2 ** int(np.ceil(np.log2(n)))
    padded_signal = np.pad(windowed_signal, (0, n_padded - n), 'constant')

    # FFT 계산
    yf = np.fft.fft(padded_signal)
    xf = np.fft.fftfreq(n_padded, 1 / fs)[:n_padded // 2]

    # 관심 주파수 범위 필터링
    idx = np.where((xf >= 0.7) & (xf <= 3.5))

    # FFT 스펙트럼에서 최대값 찾기
    fft_spectrum = 2.0 / n_padded * np.abs(yf[0:n_padded // 2])
    if len(idx[0]) == 0:
        dominant_freq_fft = 0
    else:
        max_idx = np.argmax(fft_spectrum[idx])
        dominant_freq_fft = xf[idx][max_idx]
        print(f"FFT dominant frequency: {dominant_freq_fft} Hz")

    # 웰치 방법 적용
    freqs_welch, psd = welch(signal, fs=fs, nperseg=min(256, len(signal)))
    idx_welch = np.where((freqs_welch >= 0.7) & (freqs_welch <= 3.5))
    if len(idx_welch[0]) == 0:
        dominant_freq_welch = 0
    else:
        max_idx_welch = np.argmax(psd[idx_welch])
        dominant_freq_welch = freqs_welch[idx_welch][max_idx_welch]
        print(f"Welch dominant frequency: {dominant_freq_welch} Hz")

    # 웨이블릿 변환
    freqs = np.linspace(0.7, 3.5, num=300)
    scales = pywt.central_frequency('morl') * fs / freqs
    cwt_matrix, _ = pywt.cwt(signal, scales, 'morl', sampling_period=1 / fs)

    # 파워 스펙트럼 계산
    power = (np.abs(cwt_matrix)) ** 2
    total_power = np.sum(power, axis=1)
    dominant_freq_wavelet = freqs[np.argmax(total_power)]
    print(f"Wavelet dominant frequency: {dominant_freq_wavelet} Hz")

    # 세 가지 결과 평균
    heart_rate = (dominant_freq_fft + dominant_freq_wavelet + dominant_freq_welch) / 3 * 60

    return heart_rate, xf[idx], fft_spectrum[idx], dominant_freq_fft, dominant_freq_wavelet, dominant_freq_welch

# 칼만 필터 클래스 정의
class KalmanFilter:
    def __init__(self, process_variance=1e-4, measurement_variance=1e-2):
        self.process_variance = process_variance  # 프로세스 잡음 공분산
        self.measurement_variance = measurement_variance  # 측정 잡음 공분산
        self.posteri_estimate = None  # 사후 추정값
        self.posteri_error_estimate = 1.0  # 사후 오차 추정값

    def update(self, measurement):
        if self.posteri_estimate is None:
            # 초기 추정값 설정
            self.posteri_estimate = measurement
        else:
            # 예측 단계
            priori_estimate = self.posteri_estimate
            priori_error_estimate = self.posteri_error_estimate + self.process_variance

            # 칼만 이득 계산
            kalman_gain = priori_error_estimate / (priori_error_estimate + self.measurement_variance)

            # 업데이트 단계
            self.posteri_estimate = priori_estimate + kalman_gain * (measurement - priori_estimate)
            self.posteri_error_estimate = (1 - kalman_gain) * priori_error_estimate

        return self.posteri_estimate

# Matplotlib Figure를 NumPy 배열로 변환하는 함수
def fig2data(fig):
    fig.canvas.draw()
    buf = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    buf = buf.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    return buf

def main():
    cap = cv2.VideoCapture(0)

    # 카메라의 해상도와 FPS 설정
    desired_width = 1280
    desired_height = 720
    desired_fps = 60

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, desired_width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, desired_height)
    cap.set(cv2.CAP_PROP_FPS, desired_fps)

    # 실제 FPS 가져오기
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or fps is None or np.isnan(fps):
        # FPS를 추정
        fps = desired_fps  # 또는 FPS를 정확히 얻는 방법 사용
    print(f"Using FPS: {fps}")

    # 신호 저장용 리스트 (신호 길이 연장)
    signal_length_seconds = 10  # 신호 길이 (초)
    signal_history = deque(maxlen=int(fps * signal_length_seconds))
    heart_rate_history = deque(maxlen=5)

    last_heart_rate = None
    last_display_time = time.time()
    update_interval = 2

    # Matplotlib 그래프 설정
    plt.switch_backend('agg')
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 8))  # 그래프 크기 조정

    xdata = []
    ydata = []
    peak_signal = []

    line1, = ax1.plot([], [], color='blue', label='Signal')
    line2, = ax1.plot([], [], color='red', label='Peaks')

    ax1.set_title('Green Channel Signal')
    ax1.set_xlabel('Time (frames)')
    ax1.set_ylabel('Amplitude')
    ax1.legend()

    plt.close()

    # 칼만 필터 초기화
    kalman_filter = KalmanFilter(process_variance=1e-4, measurement_variance=1e-2)

    # 랜덤 포레스트 회귀 모델 초기화
    rf_model = RandomForestRegressor(n_estimators=100)
    training_data = []
    training_labels = []
    min_training_size = 50  # 최소 학습 데이터 크기

    while True:
        ret, frame = cap.read()
        if not ret:
            continue
        frame = cv2.flip(frame, 1)
        current_time = time.perf_counter()  # 높은 정밀도의 타이머 사용

        # MediaPipe를 사용하여 얼굴 랜드마크 추출
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(frame_rgb)

        if results.multi_face_landmarks:
            # 첫 번째 얼굴만 사용
            face_landmarks = results.multi_face_landmarks[0]

            # 랜드마크 좌표 추출
            landmarks = []
            for idx, lm in enumerate(face_landmarks.landmark):
                x, y = int(lm.x * frame.shape[1]), int(lm.y * frame.shape[0])
                landmarks.append((x, y))

            # 5개의 구간에서 G 채널 신호 추출
            green_signals, regions = extract_green_signals_by_face_regions(frame, landmarks)

            # 평균 G 채널 신호 계산
            mean_green_signal = np.mean(green_signals)
            # 신호와 타임스탬프를 함께 저장
            signal_history.append((mean_green_signal, current_time))

            # 신호 길이가 충분한지 확인
            if len(signal_history) == signal_history.maxlen:
                # 신호와 타임스탬프 분리
                green_signal_array = np.array([item[0] for item in signal_history])
                time_stamps_array = np.array([item[1] for item in signal_history])
                # 시간 간격 계산
                time_intervals = np.diff(time_stamps_array)
                if np.any(time_intervals <= 0):
                    print("Warning: Non-positive time intervals detected.")
                effective_fps = 1 / np.mean(time_intervals)
                print(f"Effective FPS: {effective_fps}")
                print(f"Average time interval: {np.mean(time_intervals)}, Std dev: {np.std(time_intervals)}")

                if effective_fps < 10 or np.isnan(effective_fps):
                    print("Effective FPS too low or invalid, setting to default value.")
                    effective_fps = fps  # 또는 다른 적절한 값

                # 그린 채널 방식 적용
                processed_signal = green_channel_method(green_signal_array)

                # Bandpass Filtering 적용
                filtered_signal = bandpass_filter(processed_signal, lowcut=0.7, highcut=3.5, fs=effective_fps)

                # 신호 스무딩
                filtered_signal = smooth_signal(filtered_signal, window_length=5, polyorder=2)

                # 심박수 추정 - 피크 디텍션
                hr_peaks, peaks = calculate_heart_rate_peaks(filtered_signal, effective_fps)
                # 심박수 추정 - FFT, 웨이블릿 변환, 웰치 방법 결합
                hr_fft_wavelet, freqs, fft_spectrum, dominant_freq_fft, dominant_freq_wavelet, dominant_freq_welch = calculate_heart_rate_fft_wavelet(filtered_signal, effective_fps)
                print(f"HR Peaks = {hr_peaks:.2f} BPM, HR FFT+Wavelet+Welch = {hr_fft_wavelet:.2f} BPM")
                print(f"Dominant Frequency FFT: {dominant_freq_fft:.2f} Hz, Wavelet: {dominant_freq_wavelet:.2f} Hz, Welch: {dominant_freq_welch:.2f} Hz")

                # 결과 간의 차이 계산
                differences = [abs(hr_peaks - hr_fft_wavelet)]

                # 이전 심박수 가져오기 (없으면 초기값 설정)
                if last_heart_rate is not None:
                    prev_hr = last_heart_rate
                else:
                    prev_hr = (hr_peaks + hr_fft_wavelet) / 2

                # 머신러닝 모델을 사용하여 심박수 예측
                features = np.array([
                    hr_peaks,
                    hr_fft_wavelet,
                    dominant_freq_fft * 60,
                    dominant_freq_wavelet * 60,
                    dominant_freq_welch * 60
                ]).reshape(1, -1)
                if len(training_data) < min_training_size:
                    # 충분한 데이터가 모일 때까지 모델을 학습
                    training_data.append(features[0])
                    training_labels.append(prev_hr)
                    rf_model.fit(training_data, training_labels)
                    final_hr = prev_hr
                else:
                    # 모델을 사용하여 심박수 예측
                    final_hr = rf_model.predict(features)[0]

                    # 새로운 데이터로 모델 업데이트
                    training_data.append(features[0])
                    training_labels.append(final_hr)
                    rf_model.fit(training_data, training_labels)

                # 디버깅: 최종 심박수 출력
                print(f"Final Heart Rate: {final_hr:.2f} BPM")

                # 칼만 필터를 사용하여 시간적 일관성 반영
                filtered_heart_rate = kalman_filter.update(final_hr)

                heart_rate_history.append(filtered_heart_rate)
                smoothed_heart_rate = np.mean(heart_rate_history)

                # 그래프 업데이트
                xdata = np.arange(len(filtered_signal))
                ydata = filtered_signal
                line1.set_data(xdata, ydata)
                ax1.set_xlim(0, len(filtered_signal))
                ax1.set_ylim(np.min(ydata) - 0.5, np.max(ydata) + 0.5)

                # 피크 표시를 선 그래프로 변경
                peak_signal = np.zeros_like(ydata)
                if len(peaks) > 0:
                    peak_signal[peaks] = ydata[peaks]
                line2.set_data(xdata, peak_signal)

                # FFT 스펙트럼 그래프
                ax2.cla()  # 축 초기화
                ax2.plot(freqs * 60, fft_spectrum)
                ax2.set_xlim(40, 210)  # BPM 단위로 변경 (0.7*60 ~ 3.5*60)
                ax2.set_title('FFT Spectrum')
                ax2.set_xlabel('Frequency (BPM)')
                ax2.set_ylabel('Amplitude')

                # 그래프 업데이트
                fig.canvas.draw()
                plot_img = fig2data(fig)
                plot_img = cv2.cvtColor(plot_img, cv2.COLOR_RGB2BGR)
                cv2.imshow('Green Channel Signal and FFT Spectrum', plot_img)
                # 그래프를 초기화하지 않음

                # 2초마다 화면에 심박수 업데이트
                display_current_time = time.time()
                if display_current_time - last_display_time > update_interval:
                    last_heart_rate = smoothed_heart_rate
                    last_display_time = display_current_time

                # ROI 영역에 삼각형 표시
                display_triangular_regions(frame, regions)

                # 심박수 화면에 표시
                if last_heart_rate is not None:
                    cv2.putText(frame, f'Heart Rate: {int(last_heart_rate)} BPM', (10, 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            else:
                cv2.putText(frame, 'Collecting data...', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

        else:
            cv2.putText(frame, 'No face detected', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        cv2.imshow('rPPG Heart Rate Monitor with Green Channel', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    face_mesh.close()

if __name__ == "__main__":
    main()

        
# Applying Histogram Equalization to an example frame
frame = apply_histogram_equalization(frame)

# Assuming 'prev_frame' and 'frame' are consecutive frames in a video
optical_flow_image = compute_optical_flow(prev_frame, frame)

# Display the results for debugging
cv2.imshow('Optical Flow', optical_flow_image)
cv2.imshow('Histogram Equalized Frame', frame)
cv2.waitKey(0)
cv2.destroyAllWindows()


Using FPS: 30.0
Effective FPS: 7.9234103965288485
Average time interval: 0.12620828026755854, Std dev: 0.01818180836966474
Effective FPS too low or invalid, setting to default value.
Bandpass Filter - fs: 30.0, nyquist: 15.0, low: 0.04666666666666666, high: 0.23333333333333334
FFT dominant frequency: 0.87890625 Hz
Welch dominant frequency: 0.8203125 Hz
Wavelet dominant frequency: 0.9622073578595317 Hz
HR Peaks = 76.33 BPM, HR FFT+Wavelet+Welch = 53.23 BPM
Dominant Frequency FFT: 0.88 Hz, Wavelet: 0.96 Hz, Welch: 0.82 Hz
Final Heart Rate: 64.78 BPM
Effective FPS: 7.833808257303931
Average time interval: 0.12765183511705686, Std dev: 0.03777058859260982
Effective FPS too low or invalid, setting to default value.
Bandpass Filter - fs: 30.0, nyquist: 15.0, low: 0.04666666666666666, high: 0.23333333333333334
FFT dominant frequency: 0.87890625 Hz
Welch dominant frequency: 0.8203125 Hz
Wavelet dominant frequency: 0.8966555183946487 Hz
HR Peaks = 71.48 BPM, HR FFT+Wavelet+Welch = 51.92 BPM
Dom

NameError: name 'apply_histogram_equalization' is not defined